In [174]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from PIL import Image
import json

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torch.optim as optim
from torchvision.transforms import v2
from torchvision import models
from tqdm import tqdm

In [175]:
def create_json(path):
    dirs = list(os.listdir(os.path.join(os.curdir, path)))
    json_format = dict()
    for i in range(len(dirs)):
        json_format[str(dirs[i])] = i
    global NUM_CLASSES
    NUM_CLASSES = len(dirs)
    with open(os.path.join(os.curdir, path, "format.json"), 'w') as file:
        json.dump(json_format, file)

create_json('data')

In [176]:
class FruitsDataset(Dataset):
    def __init__(self, work_dir, transform = None):
        super().__init__()
        self.files = []
        self.length = 0
        self.path = os.path.join(os.curdir, work_dir)
        with open(os.path.join(self.path, 'format.json'), 'r') as file:
            self.format = json.load(file)
        
        self.transform = transform
        self.targets = torch.eye(10)
        for _dir, _target in self.format.items():
            print(os.path.join(self.path, _dir))
            files_path = os.path.join(self.path, _dir)
            _files = os.listdir(files_path)
            self.length += len(_files)
            self.files.extend(map(lambda _x: (os.path.join(files_path, _x), _target), _files))
    def __len__(self):
        return self.length
    def __getitem__(self, item):
        path, target = self.files[item]
        img = Image.open(path).convert('RGB') 
        
        if self.transform:
            img = self.transform(img)
        
        return img, int(target)

In [177]:
IMG_SIZE = 256

transforms = v2.Compose([
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.RandomHorizontalFlip(),
    v2.RandomRotation(15),
    v2.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


In [178]:
fruits_data = FruitsDataset('data', transform = transforms)

.\data\FreshApple
.\data\FreshBanana
.\data\FreshCapsicum
.\data\FreshCucumber
.\data\FreshPotato
.\data\RottenApple
.\data\RottenBanana
.\data\RottenCapsicum
.\data\RottenCucumber
.\data\RottenPotato


In [179]:
train_size = train_size = int(len(fruits_data) * 0.7)
test_size = len(fruits_data) - train_size

In [180]:
traindata, valdata = random_split(fruits_data, [train_size, test_size], generator=torch.Generator().manual_seed(42))

In [181]:
train_batch = DataLoader(traindata, batch_size = 64, shuffle = True)
val_batch = DataLoader(valdata, batch_size = 64, shuffle = False)

In [202]:
class MyCV(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, 4, padding='same'), 
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 32, 4, padding='same'), 
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 16, 4, padding='same'), 
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 8, 4, padding='same'), 
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.model(x)

In [203]:
model = MyCV()

In [207]:
optimizer = optim.Adam(params=model.parameters(), lr=0.001, weight_decay=0.001)
loss_function = nn.CrossEntropyLoss()

In [209]:
epochs = 5
model.train()
 
for _e in range(epochs):
    loss_mean = 0
    lm_count = 0
 
    train_tqdm = tqdm(train_batch, leave=True)
    for x_train, y_train in train_tqdm:
        predict = model(x_train)
        loss = loss_function(predict, y_train)
 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
 
        lm_count += 1
        loss_mean = 1/lm_count * loss.item() + (1 - 1/lm_count) * loss_mean
        train_tqdm.set_description(f"Epoch [{_e+1}/{epochs}], loss_mean={loss_mean:.3f}")

Epoch [5/5], loss_mean=0.604: 100%|██████████████████████████████████████████████████| 139/139 [09:02<00:00,  3.90s/it]


In [210]:
st = model.state_dict()
torch.save(st, 'model_1.tar')

In [211]:
Q = 0
count = 0
correct_predictions = 0
total_samples = 0

model.eval() 

val_tqdm = tqdm(val_batch, leave=True)

with torch.no_grad(): 
    for x_val, y_val in val_tqdm:
        p = model(x_val)  
        
        loss = loss_function(p, y_val)
        Q += loss.item()
        
        _, predicted = torch.max(p.data, 1)
        correct_predictions += (predicted == y_val).sum().item()
        total_samples += y_val.size(0)
        
        count += 1

Q /= count
accuracy = correct_predictions / total_samples

print(f"Validation Loss: {Q:.4f}")
print(f"Validation Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

100%|██████████████████████████████████████████████████████████████████████████████████| 60/60 [02:03<00:00,  2.05s/it]

Validation Loss: 0.6116
Validation Accuracy: 0.7888 (78.88%)
